# 🏁 NEXUS Stock AI — Phases 31, 32, & 33
## Automated Pipeline Testing, Reproducibility Lockfile, & Final Documentation

**Objective:** Complete the final milestone of the NEXUS Stock AI engineering roadmap by implementing automated leakage and integrity tests (`pytest`), creating the environment lockfile (`requirements.txt`), generating production documentation (`README.md`), and finalizing the deployment backup archive.

### 🔬 Phases Implemented:
1. **Phase 31 (Automated Leakage & Integrity Testing):**
   - **Leakage Check 1:** Verify no target or forward-return variables exist in `feature_schema.json`.
   - **Leakage Check 2:** Strict assertion that training (2018–2021) and test (2023) windows have zero temporal overlap.
   - **Data Integrity:** Confirm zero duplicate `(stock_symbol, date)` records exist.
   - **Engine Verification:** Test standalone `NexusInferenceEngine` and `VectorizedBacktester`.
2. **Phase 32 (Reproducibility & Environment Packaging):**
   - Generate clean `requirements.txt` containing all production ML and UI dependencies.
3. **Phase 33 (Comprehensive Documentation & Packaging):**
   - Generate full root `README.md` with system architecture diagrams, research findings, and setup guides.
   - Finalize `nexus_data_backup.zip` containing all models, code, tests, and documentation.

### ⚙️ Step 1: Environment Setup & Google Drive Mount

In [1]:
# Install test dependencies
%pip install -q pytest pyarrow xgboost scikit-learn joblib shap streamlit plotly

import os
import sys
import time
import json
import shutil
import zipfile
import base64

# Mount Google Drive if in Colab
try:
    from google.colab import drive
    print("Mounting Google Drive at /content/drive...")
    drive.mount("/content/drive")
    print("✓ Google Drive mounted successfully.")
except Exception as e:
    print(f"Drive mount note: {e}")

GDRIVE_BASE = "/content/drive/MyDrive/NEXUS_Stock_AI"
print(f"Drive Base Path: {GDRIVE_BASE} (Exists: {os.path.exists(GDRIVE_BASE)})")


Mounting Google Drive at /content/drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted successfully.
Drive Base Path: /content/drive/MyDrive/NEXUS_Stock_AI (Exists: True)


### 🧪 Step 2: Auto-Scaffold Test Suite (`tests/test_pipeline.py`)
Generates the automated testing suite and requirements lockfile.

In [2]:
print("=" * 80)
print("PHASE 31: AUTO-SCAFFOLDING TEST SUITE & REQUIREMENTS")
print("=" * 80)

# 1. Create tests directory
os.makedirs("tests", exist_ok=True)

test_init_b64 = "IyBUZXN0IHBhY2thZ2UgaW5pdGlhbGl6YXRpb24K"
with open("tests/__init__.py", "wb") as f:
    f.write(base64.b64decode(test_init_b64))

test_pipeline_b64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKTkVYVVMgU3RvY2sgQUkg4oCUIEF1dG9tYXRlZCBQaXBlbGluZSwgSW50ZWdyaXR5LCAmIExlYWthZ2UgVGVzdCBTdWl0ZSAoUGhhc2VzIDMxICYgMzIpClN0cmljdCBhc3NlcnRpb25zIHZlcmlmeWluZyBkYXRhIGxlYWthZ2UgcHJldmVudGlvbiwgdGVtcG9yYWwgZGlzam9pbnRuZXNzLAptb2RlbCBhcnRpZmFjdCBpbnRlZ3JpdHksIGFuZCBzdGFuZGFsb25lIGluZmVyZW5jZS4KIiIiCgppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQganNvbgp0cnk6CiAgICBpbXBvcnQgcHl0ZXN0CmV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgIGNsYXNzIE1vY2tQeXRlc3Q6CiAgICAgICAgQHN0YXRpY21ldGhvZAogICAgICAgIGRlZiBza2lwKHJlYXNvbj0iIik6CiAgICAgICAgICAgIHByaW50KGYiU0tJUFBFRDoge3JlYXNvbn0iKQogICAgcHl0ZXN0ID0gTW9ja1B5dGVzdCgpCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKIyBBZGQgcmVwb3NpdG9yeSByb290IHRvIHBhdGgKUk9PVF9ESVIgPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXyksICIuLiIpKQppZiBST09UX0RJUiBub3QgaW4gc3lzLnBhdGg6CiAgICBzeXMucGF0aC5pbnNlcnQoMCwgUk9PVF9ESVIpCgoKZGVmIHJlc29sdmVfZmlsZShmaWxlbmFtZTogc3RyLCBzdWJmb2xkZXI6IHN0ciA9ICJtb2RlbHMiKSAtPiBzdHI6CiAgICAiIiJGaW5kcyBhbiBhcnRpZmFjdCBsb2NhbGx5IG9yIG9uIEdvb2dsZSBEcml2ZS4iIiIKICAgIGNhbmRpZGF0ZXMgPSBbCiAgICAgICAgb3MucGF0aC5qb2luKFJPT1RfRElSLCBzdWJmb2xkZXIsIGZpbGVuYW1lKSwKICAgICAgICBvcy5wYXRoLmpvaW4oUk9PVF9ESVIsIGZpbGVuYW1lKSwKICAgICAgICBvcy5wYXRoLmpvaW4oIi9jb250ZW50L2RyaXZlL015RHJpdmUvTkVYVVNfU3RvY2tfQUkiLCBzdWJmb2xkZXIsIGZpbGVuYW1lKSwKICAgICAgICBvcy5wYXRoLmpvaW4oIi9jb250ZW50Iiwgc3ViZm9sZGVyLCBmaWxlbmFtZSkKICAgIF0KICAgIGZvciBjIGluIGNhbmRpZGF0ZXM6CiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoYyk6CiAgICAgICAgICAgIHJldHVybiBjCiAgICByZXR1cm4gY2FuZGlkYXRlc1swXQoKCmRlZiB0ZXN0X2xlYWthZ2VfY2hlY2tfMV9ub190YXJnZXRfaW5fZmVhdHVyZV9zY2hlbWEoKToKICAgICIiIgogICAgTGVha2FnZSBDaGVjayAxOgogICAgQXNzZXJ0IHRoYXQgbmVpdGhlciAndGFyZ2V0JywgJ2ZvcndhcmRfcmV0dXJuJywgbm9yIGFueSBmdXR1cmUgbG9va2FoZWFkCiAgICB2YXJpYWJsZSBpcyBwcmVzZW50IGluIGZlYXR1cmVfc2NoZW1hLmpzb24uCiAgICAiIiIKICAgIHNjaGVtYV9wYXRoID0gcmVzb2x2ZV9maWxlKCJmZWF0dXJlX3NjaGVtYS5qc29uIiwgc3ViZm9sZGVyPSJtb2RlbHMiKQogICAgYXNzZXJ0IG9zLnBhdGguZXhpc3RzKHNjaGVtYV9wYXRoKSwgZiJmZWF0dXJlX3NjaGVtYS5qc29uIG5vdCBmb3VuZCBhdCB7c2NoZW1hX3BhdGh9IgoKICAgIHdpdGggb3BlbihzY2hlbWFfcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHNjaGVtYSA9IGpzb24ubG9hZChmKQoKICAgIGZlYXR1cmVzID0gc2NoZW1hWyJmZWF0dXJlcyJdIGlmIGlzaW5zdGFuY2Uoc2NoZW1hLCBkaWN0KSBlbHNlIHNjaGVtYQogICAgZm9yYmlkZGVuX3Rlcm1zID0gWwogICAgICAgICJ0YXJnZXQiLCAiZm9yd2FyZF9yZXR1cm4iLCAiZm9yd2FyZF9yZXR1cm5fMWQiLAogICAgICAgICJuZXh0X2Nsb3NlIiwgImZ1dHVyZSIsICJsZWFkIiwgImxhYmVsIiwgIm91dGNvbWUiCiAgICBdCgogICAgZm9yIGZlYXQgaW4gZmVhdHVyZXM6CiAgICAgICAgZmVhdF9sb3dlciA9IGZlYXQubG93ZXIoKQogICAgICAgIGZvciBmb3JiaWRkZW4gaW4gZm9yYmlkZGVuX3Rlcm1zOgogICAgICAgICAgICBhc3NlcnQgZm9yYmlkZGVuICE9IGZlYXRfbG93ZXIsIGYiQ1JJVElDQUwgTEVBS0FHRTogRm9yYmlkZGVuIHZhcmlhYmxlICd7ZmVhdH0nIGZvdW5kIGluIGZlYXR1cmUgc2NoZW1hISIKCiAgICBhc3NlcnQgbGVuKGZlYXR1cmVzKSA9PSAyMywgZiJFeHBlY3RlZCAyMyBzZXF1ZW50aWFsIGZlYXR1cmVzLCBmb3VuZCB7bGVuKGZlYXR1cmVzKX0iCgoKZGVmIHRlc3RfbGVha2FnZV9jaGVja18yX3RlbXBvcmFsX3dpbmRvd19kaXNqb2ludG5lc3MoKToKICAgICIiIgogICAgTGVha2FnZSBDaGVjayAyOgogICAgQXNzZXJ0IHRoYXQgdGhlIHRyYWluaW5nIGRhdGUgd2luZG93IGFuZCBvdXQtb2Ytc2FtcGxlIHRlc3QgZGF0ZSB3aW5kb3cKICAgIGluIG1vZGVsX21ldGFkYXRhLmpzb24gc3RyaWN0bHkgZG8gbm90IG92ZXJsYXAuCiAgICAiIiIKICAgIG1ldGFfcGF0aCA9IHJlc29sdmVfZmlsZSgibW9kZWxfbWV0YWRhdGEuanNvbiIsIHN1YmZvbGRlcj0ibW9kZWxzIikKICAgIGFzc2VydCBvcy5wYXRoLmV4aXN0cyhtZXRhX3BhdGgpLCBmIm1vZGVsX21ldGFkYXRhLmpzb24gbm90IGZvdW5kIGF0IHttZXRhX3BhdGh9IgoKICAgIHdpdGggb3BlbihtZXRhX3BhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBtZXRhZGF0YSA9IGpzb24ubG9hZChmKQoKICAgIHRyYWluX2VuZCA9IHBkLnRvX2RhdGV0aW1lKG1ldGFkYXRhWyJ0cmFpbmluZ193aW5kb3ciXVsiZW5kIl0pCiAgICB2YWxfc3RhcnQgPSBwZC50b19kYXRldGltZShtZXRhZGF0YVsidmFsaWRhdGlvbl93aW5kb3ciXVsic3RhcnQiXSkKICAgIHZhbF9lbmQgPSBwZC50b19kYXRldGltZShtZXRhZGF0YVsidmFsaWRhdGlvbl93aW5kb3ciXVsiZW5kIl0pCiAgICB0ZXN0X3N0YXJ0ID0gcGQudG9fZGF0ZXRpbWUobWV0YWRhdGFbInRlc3Rfd2luZG93Il1bInN0YXJ0Il0pCgogICAgIyBTdHJpY3QgY2hyb25vbG9naWNhbCBvcmRlcmluZzogVHJhaW4gRW5kIDwgVmFsIFN0YXJ0IDw9IFZhbCBFbmQgPCBUZXN0IFN0YXJ0CiAgICBhc3NlcnQgdHJhaW5fZW5kIDwgdmFsX3N0YXJ0LCBmIlRyYWluIEVuZCAoe3RyYWluX2VuZH0pIG92ZXJsYXBzIHdpdGggVmFsIFN0YXJ0ICh7dmFsX3N0YXJ0fSkiCiAgICBhc3NlcnQgdmFsX2VuZCA8IHRlc3Rfc3RhcnQsIGYiVmFsIEVuZCAoe3ZhbF9lbmR9KSBvdmVybGFwcyB3aXRoIFRlc3QgU3RhcnQgKHt0ZXN0X3N0YXJ0fSkiCiAgICBhc3NlcnQgdHJhaW5fZW5kIDwgdGVzdF9zdGFydCwgZiJDUklUSUNBTCBMRUFLQUdFOiBUcmFpbiBFbmQgKHt0cmFpbl9lbmR9KSA+PSBUZXN0IFN0YXJ0ICh7dGVzdF9zdGFydH0pIgoKCmRlZiB0ZXN0X2RhdGFfaW50ZWdyaXR5X25vX2R1cGxpY2F0ZV9yb3dzKCk6CiAgICAiIiIKICAgIERhdGEgSW50ZWdyaXR5OgogICAgQXNzZXJ0IHRoYXQgcHJpY2VzIGRhdGEgY29udGFpbnMgbm8gZHVwbGljYXRlIChzdG9ja19zeW1ib2wsIGRhdGUpIHJvd3MuCiAgICAiIiIKICAgIHByaWNlX3BhdGggPSByZXNvbHZlX2ZpbGUoImNsZWFuZWRfcHJpY2VzLnBhcnF1ZXQiLCBzdWJmb2xkZXI9ImRhdGEiKQogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKHByaWNlX3BhdGgpOgogICAgICAgIHByaWNlX3BhdGggPSByZXNvbHZlX2ZpbGUoIm1vZGVsX2RhdGFzZXRfdjEucGFycXVldCIsIHN1YmZvbGRlcj0iZGF0YSIpCgogICAgaWYgb3MucGF0aC5leGlzdHMocHJpY2VfcGF0aCk6CiAgICAgICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQocHJpY2VfcGF0aCwgY29sdW1ucz1bInN0b2NrX3N5bWJvbCIsICJkYXRlIl0pCiAgICAgICAgZGZbImRhdGUiXSA9IHBkLnRvX2RhdGV0aW1lKGRmWyJkYXRlIl0pLmR0Lm5vcm1hbGl6ZSgpCiAgICAgICAgZHVwX2NvdW50ID0gZGYuZHVwbGljYXRlZChzdWJzZXQ9WyJzdG9ja19zeW1ib2wiLCAiZGF0ZSJdKS5zdW0oKQogICAgICAgIGFzc2VydCBkdXBfY291bnQgPT0gMCwgZiJGb3VuZCB7ZHVwX2NvdW50fSBkdXBsaWNhdGUgKHN0b2NrX3N5bWJvbCwgZGF0ZSkgcmVjb3JkcyBpbiB7cHJpY2VfcGF0aH0hIgogICAgZWxzZToKICAgICAgICBweXRlc3Quc2tpcCgiUHJpY2UgcGFycXVldCBmaWxlIG5vdCBmb3VuZCBpbiBjdXJyZW50IGVudmlyb25tZW50IChhdmFpbGFibGUgb24gR29vZ2xlIERyaXZlKS4iKQoKCmRlZiB0ZXN0X21vZGVsX2FydGlmYWN0X2V4aXN0ZW5jZSgpOgogICAgIiIiCiAgICBNb2RlbCBBcnRpZmFjdHM6CiAgICBWZXJpZnkgdGhhdCBjYWxpYnJhdGVkIG9yIG5hdGl2ZSBYR0Jvb3N0IHdlaWdodHMgZXhpc3QuCiAgICAiIiIKICAgIGNhbGlicmF0ZWRfam9ibGliID0gcmVzb2x2ZV9maWxlKCJ4Z2JfZGlyZWN0aW9uX2NhbGlicmF0ZWQuam9ibGliIiwgc3ViZm9sZGVyPSJtb2RlbHMiKQogICAgbmF0aXZlX2pzb24gPSByZXNvbHZlX2ZpbGUoInhnYl9kaXJlY3Rpb24uanNvbiIsIHN1YmZvbGRlcj0ibW9kZWxzIikKCiAgICBoYXNfbW9kZWwgPSBvcy5wYXRoLmV4aXN0cyhjYWxpYnJhdGVkX2pvYmxpYikgb3Igb3MucGF0aC5leGlzdHMobmF0aXZlX2pzb24pCiAgICBhc3NlcnQgaGFzX21vZGVsLCAiTmVpdGhlciB4Z2JfZGlyZWN0aW9uX2NhbGlicmF0ZWQuam9ibGliIG5vciB4Z2JfZGlyZWN0aW9uLmpzb24gd2FzIGZvdW5kIGluIG1vZGVscy8hIgoKCmRlZiB0ZXN0X3N0YW5kYWxvbmVfaW5mZXJlbmNlX2VuZ2luZV9leGVjdXRpb24oKToKICAgICIiIgogICAgSW5mZXJlbmNlIEVuZ2luZToKICAgIFZlcmlmeSB0aGF0IE5leHVzSW5mZXJlbmNlRW5naW5lIGxvYWRzIGFuZCBleGVjdXRlcyB3aXRoIHZhbGlkIGR1bW15IGlucHV0cy4KICAgICIiIgogICAgZnJvbSBzcmMuaW5mZXJlbmNlLnByZWRpY3QgaW1wb3J0IE5leHVzSW5mZXJlbmNlRW5naW5lCgogICAgc2NoZW1hX3BhdGggPSByZXNvbHZlX2ZpbGUoImZlYXR1cmVfc2NoZW1hLmpzb24iLCBzdWJmb2xkZXI9Im1vZGVscyIpCiAgICB3aXRoIG9wZW4oc2NoZW1hX3BhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBzY2hlbWEgPSBqc29uLmxvYWQoZikKICAgIGZlYXR1cmVzID0gc2NoZW1hWyJmZWF0dXJlcyJdIGlmIGlzaW5zdGFuY2Uoc2NoZW1hLCBkaWN0KSBlbHNlIHNjaGVtYQoKICAgIGVuZ2luZSA9IE5leHVzSW5mZXJlbmNlRW5naW5lKHNjaGVtYV9wYXRoPXNjaGVtYV9wYXRoKQoKICAgICMgU3ludGhldGljIHNpbmdsZS1yb3cgdGVzdCBEYXRhRnJhbWUKICAgIHN5bnRoZXRpY19kYXRhID0ge2ZlYXQ6IFswLjAxIGlmICJyZXR1cm4iIGluIGZlYXQgZWxzZSA1MC4wXSBmb3IgZmVhdCBpbiBmZWF0dXJlc30KICAgIHN5bnRoZXRpY19kYXRhWyJzdG9ja19zeW1ib2wiXSA9IFsiQUFQTCJdCiAgICBzeW50aGV0aWNfZGF0YVsiZGF0ZSJdID0gWyIyMDIzLTA2LTE1Il0KICAgIHRlc3RfZGYgPSBwZC5EYXRhRnJhbWUoc3ludGhldGljX2RhdGEpCgogICAgcmVzdWx0cyA9IGVuZ2luZS5wcmVkaWN0KHRlc3RfZGYsIHRvcF9rX2ZhY3RvcnM9MykKICAgIGFzc2VydCBsZW4ocmVzdWx0cykgPT0gMQogICAgYXNzZXJ0IHJlc3VsdHNbMF1bInRpY2tlciJdID09ICJBQVBMIgogICAgYXNzZXJ0IHJlc3VsdHNbMF1bInByZWRpY3Rpb24iXSBpbiBbIlVQIiwgIkRPV04iXQogICAgYXNzZXJ0IDAuMCA8PSByZXN1bHRzWzBdWyJ1cF9wcm9iYWJpbGl0eSJdIDw9IDEuMAogICAgYXNzZXJ0IDAuMCA8PSByZXN1bHRzWzBdWyJkb3duX3Byb2JhYmlsaXR5Il0gPD0gMS4wCiAgICBhc3NlcnQgbGVuKHJlc3VsdHNbMF1bInRvcF9mYWN0b3JzIl0pID4gMAoKCmRlZiB0ZXN0X3ZlY3Rvcml6ZWRfYmFja3Rlc3Rlcl9leGVjdXRpb24oKToKICAgICIiIgogICAgQmFja3Rlc3RpbmcgRW5naW5lOgogICAgVmVyaWZ5IHRoYXQgVmVjdG9yaXplZEJhY2t0ZXN0ZXIgY29tcHV0ZXMgcmV0dXJucyBhbmQgZHJhd2Rvd24gY29ycmVjdGx5LgogICAgIiIiCiAgICBmcm9tIHNyYy5iYWNrdGVzdC5lbmdpbmUgaW1wb3J0IFZlY3Rvcml6ZWRCYWNrdGVzdGVyCgogICAgZW5naW5lID0gVmVjdG9yaXplZEJhY2t0ZXN0ZXIoY29zdF9icHM9MTAuMCkKCiAgICAjIFN5bnRoZXRpYyA1LWRheSAyLXN0b2NrIHRlc3QgZGF0YXNldAogICAgZGF0ZXMgPSBwZC5kYXRlX3JhbmdlKCIyMDIzLTAxLTAxIiwgcGVyaW9kcz01KQogICAgcmVjb3JkcyA9IFtdCiAgICBmb3IgZCBpbiBkYXRlczoKICAgICAgICBmb3Igc3ltIGluIFsiQUFQTCIsICJNU0ZUIl06CiAgICAgICAgICAgIHJlY29yZHMuYXBwZW5kKHsKICAgICAgICAgICAgICAgICJkYXRlIjogZCwKICAgICAgICAgICAgICAgICJzdG9ja19zeW1ib2wiOiBzeW0sCiAgICAgICAgICAgICAgICAiY2xvc2UiOiAxNTAuMCwKICAgICAgICAgICAgICAgICJmb3J3YXJkX3JldHVybl8xZCI6IDAuMDEsCiAgICAgICAgICAgICAgICAidXBfcHJvYmFiaWxpdHkiOiAwLjU1CiAgICAgICAgICAgIH0pCiAgICBtb2NrX2RmID0gcGQuRGF0YUZyYW1lKHJlY29yZHMpCgogICAgcmVzdWx0cyA9IGVuZ2luZS5ydW5fYmFja3Rlc3QobW9ja19kZiwgcHJvYl9jb2w9InVwX3Byb2JhYmlsaXR5IiwgdGhyZXNob2xkPTAuNTApCiAgICBhc3NlcnQgInN0cmF0X21ldHJpY3MiIGluIHJlc3VsdHMKICAgIGFzc2VydCAicG9ydGZvbGlvX2RhaWx5IiBpbiByZXN1bHRzCiAgICBhc3NlcnQgcmVzdWx0c1sic3RyYXRfbWV0cmljcyJdWyJjdW11bGF0aXZlX3JldHVybiJdID4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwcmludCgiPSIgKiA2MCkKICAgIHByaW50KCJSVU5OSU5HIE5FWFVTIFNUT0NLIEFJIFBJUEVMSU5FIElOVEVHUklUWSBURVNUUyIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIHRlc3RfbGVha2FnZV9jaGVja18xX25vX3RhcmdldF9pbl9mZWF0dXJlX3NjaGVtYSgpCiAgICBwcmludCgi4pyTIFtURVNUIDEvNiBQQVNTRURdIHRlc3RfbGVha2FnZV9jaGVja18xX25vX3RhcmdldF9pbl9mZWF0dXJlX3NjaGVtYSIpCiAgICB0ZXN0X2xlYWthZ2VfY2hlY2tfMl90ZW1wb3JhbF93aW5kb3dfZGlzam9pbnRuZXNzKCkKICAgIHByaW50KCLinJMgW1RFU1QgMi82IFBBU1NFRF0gdGVzdF9sZWFrYWdlX2NoZWNrXzJfdGVtcG9yYWxfd2luZG93X2Rpc2pvaW50bmVzcyIpCiAgICB0ZXN0X2RhdGFfaW50ZWdyaXR5X25vX2R1cGxpY2F0ZV9yb3dzKCkKICAgIHByaW50KCLinJMgW1RFU1QgMy82IFBBU1NFRF0gdGVzdF9kYXRhX2ludGVncml0eV9ub19kdXBsaWNhdGVfcm93cyIpCiAgICB0ZXN0X21vZGVsX2FydGlmYWN0X2V4aXN0ZW5jZSgpCiAgICBwcmludCgi4pyTIFtURVNUIDQvNiBQQVNTRURdIHRlc3RfbW9kZWxfYXJ0aWZhY3RfZXhpc3RlbmNlIikKICAgIHRlc3Rfc3RhbmRhbG9uZV9pbmZlcmVuY2VfZW5naW5lX2V4ZWN1dGlvbigpCiAgICBwcmludCgi4pyTIFtURVNUIDUvNiBQQVNTRURdIHRlc3Rfc3RhbmRhbG9uZV9pbmZlcmVuY2VfZW5naW5lX2V4ZWN1dGlvbiIpCiAgICB0ZXN0X3ZlY3Rvcml6ZWRfYmFja3Rlc3Rlcl9leGVjdXRpb24oKQogICAgcHJpbnQoIuKckyBbVEVTVCA2LzYgUEFTU0VEXSB0ZXN0X3ZlY3Rvcml6ZWRfYmFja3Rlc3Rlcl9leGVjdXRpb24iKQogICAgcHJpbnQoIlxu4pyTIEFMTCA2IFRFU1RTIFBBU1NFRCBTVUNDRVNTRlVMTFkhIikK"
with open("tests/test_pipeline.py", "wb") as f:
    f.write(base64.b64decode(test_pipeline_b64))
print(f"✓ Created tests/test_pipeline.py ({os.path.getsize('tests/test_pipeline.py')} bytes)")

# 2. Create requirements.txt
requirements_b64 = "IyBORVhVUyBTdG9jayBBSSDigJQgUHJvZHVjdGlvbiAmIFJlc2VhcmNoIERlcGVuZGVuY2llcwpwYW5kYXM+PTIuMC4wCm51bXB5Pj0xLjI0LjAKcHlhcnJvdz49MTIuMC4wCnNjaWtpdC1sZWFybj49MS4yLjAKeGdib29zdD49Mi4wLjAKam9ibGliPj0xLjMuMApvcHR1bmE+PTMuMi4wCnNoYXA+PTAuNDIuMAp0cmFuc2Zvcm1lcnM+PTQuMzAuMAp0b3JjaD49Mi4wLjAKc3RyZWFtbGl0Pj0xLjM1LjAKcGxvdGx5Pj01LjE1LjAKbWF0cGxvdGxpYj49My43LjAKc2VhYm9ybj49MC4xMi4wCnB5dGVzdD49Ny4zLjAK"
with open("requirements.txt", "wb") as f:
    f.write(base64.b64decode(requirements_b64))
print(f"✓ Created requirements.txt ({os.path.getsize('requirements.txt')} bytes)")

# Sync to Google Drive if present
if os.path.exists(GDRIVE_BASE):
    os.makedirs(os.path.join(GDRIVE_BASE, "tests"), exist_ok=True)
    shutil.copy2("tests/test_pipeline.py", os.path.join(GDRIVE_BASE, "tests", "test_pipeline.py"))
    shutil.copy2("tests/__init__.py", os.path.join(GDRIVE_BASE, "tests", "__init__.py"))
    shutil.copy2("requirements.txt", os.path.join(GDRIVE_BASE, "requirements.txt"))
    print("✓ Synced test suite and requirements to Google Drive.")


PHASE 31: AUTO-SCAFFOLDING TEST SUITE & REQUIREMENTS
✓ Created tests/test_pipeline.py (7572 bytes)
✓ Created requirements.txt (291 bytes)
✓ Synced test suite and requirements to Google Drive.


### 🚀 Step 3: Execute Automated Pytest Test Suite
Runs all leak-prevention, temporal disjointness, and engine execution checks.

In [3]:
print("=" * 80)
print("EXECUTING PYTEST SUITE (PHASES 31 & 32)")
print("=" * 80)

!pytest tests/ -v


EXECUTING PYTEST SUITE (PHASES 31 & 32)
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.12.1, anyio-4.14.2
collected 6 items                                                              

tests/test_pipeline.py::test_leakage_check_1_no_target_in_feature_schema PASSED [ 16%]
tests/test_pipeline.py::test_leakage_check_2_temporal_window_disjointness PASSED [ 33%]
tests/test_pipeline.py::test_data_integrity_no_duplicate_rows PASSED     [ 50%]
tests/test_pipeline.py::test_model_artifact_existence PASSED             [ 66%]
tests/test_pipeline.py::test_standalone_inference_engine_execution PASSED [ 83%]
tests/test_pipeline.py::test_vectorized_backtester_execution PASSED      [100%]

============================== 6 passed in 3.55s ===============================


### 📖 Step 4: Phase 33 — Generate Comprehensive Project Documentation (`README.md`)

In [4]:
print("=" * 80)
print("PHASE 33: GENERATING PRODUCTION README.md")
print("=" * 80)

readme_b64 = "IyDwn5OIIE5FWFVTIFN0b2NrIEFJIOKAlCBNdWx0aS1Nb2RhbCBEaXJlY3Rpb25hbCBQcmVkaWN0b3IKClshW1B5dGhvbiAzLjEwK10oaHR0cHM6Ly9pbWcuc2hpZWxkcy5pby9iYWRnZS9weXRob24tMy4xMCUyQi1ibHVlLnN2ZyldKGh0dHBzOi8vd3d3LnB5dGhvbi5vcmcvKQpbIVtYR0Jvb3N0XShodHRwczovL2ltZy5zaGllbGRzLmlvL2JhZGdlL01MLVhHQm9vc3Qtb3JhbmdlLnN2ZyldKGh0dHBzOi8veGdib29zdC5haS8pClshW0ZpbkJFUlRdKGh0dHBzOi8vaW1nLnNoaWVsZHMuaW8vYmFkZ2UvTkxQLVByb3N1c0FJJTJGRmluQkVSVC15ZWxsb3cuc3ZnKV0oaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9Qcm9zdXNBSS9maW5iZXJ0KQpbIVtTdHJlYW1saXRdKGh0dHBzOi8vaW1nLnNoaWVsZHMuaW8vYmFkZ2UvVUktU3RyZWFtbGl0LXJlZC5zdmcpXShodHRwczovL3N0cmVhbWxpdC5pby8pClshW0xpY2Vuc2U6IE1JVF0oaHR0cHM6Ly9pbWcuc2hpZWxkcy5pby9iYWRnZS9MaWNlbnNlLU1JVC1ncmVlbi5zdmcpXShMSUNFTlNFKQoKLS0tCgojIyDwn4+b77iPIEV4ZWN1dGl2ZSBTdW1tYXJ5CgoqKk5FWFVTIFN0b2NrIEFJKiogaXMgYW4gaW5zdGl0dXRpb25hbC1ncmFkZSwgbXVsdGktbW9kYWwgbWFjaGluZSBsZWFybmluZyBmcmFtZXdvcmsgZGVzaWduZWQgZm9yIG5leHQtZGF5IGRpcmVjdGlvbmFsIHN0b2NrIHByZWRpY3Rpb24uIEJ5IHN5c3RlbWF0aWNhbGx5IHN5bmNocm9uaXppbmcgaGlnaC1mcmVxdWVuY3kgcHJpY2UgYWN0aW9uIHdpdGggZGVlcCBmaW5hbmNpYWwgTkxQLCBORVhVUyBicmlkZ2VzIHRoZSBnYXAgYmV0d2VlbiBxdWFudGl0YXRpdmUgdGVjaG5pY2FsIGluZGljYXRvcnMgYW5kIG5hdHVyYWwgbGFuZ3VhZ2Ugc2VudGltZW50LgoKRmluYW5jaWFsIG1hcmtldHMgb3BlcmF0ZSB3aXRoIGFzeW5jaHJvbm91cyBuZXdzIHJlbGVhc2VzLCB3ZWVrZW5kIGRpc2Nsb3N1cmVzLCBhbmQgYWZ0ZXItaG91cnMgZWFybmluZ3MgcmVwb3J0cy4gTkVYVVMgZW5mb3JjZXMgYSAqKnN0cmljdCA0OjAwIFBNIFVTL0Vhc3Rlcm4gbWFya2V0IGN1dG9mZiBydWxlKio6IGhlYWRsaW5lcyBwdWJsaXNoZWQgYmV0d2VlbiBtYXJrZXQgY2xvc2VzIGFyZSBkZXRlcm1pbmlzdGljYWxseSBhbGlnbmVkIHRvIHRoZSB0YXJnZXQgdHJhZGluZyBzZXNzaW9uIHdpdGhvdXQgbG9va2FoZWFkIGJpYXMuIERhaWx5IHNlbnRpbWVudCBzaWduYWxzIGV4dHJhY3RlZCB2aWEgKipGaW5CRVJUKiogYXJlIGZ1c2VkIHdpdGggbXVsdGktc2NhbGUgbW9tZW50dW0gYW5kIHZvbGF0aWxpdHkgaW5kaWNhdG9ycyB0byBmb3JlY2FzdCBuZXh0LWRheSBlcXVpdHkgZGlyZWN0aW9uIGFjcm9zcyA5IG1lZ2EtY2FwIGVxdWl0aWVzIChgQUFQTGAsIGBBTURgLCBgQU1aTmAsIGBHT09HTGAsIGBKUE1gLCBgTVNGVGAsIGBORkxYYCwgYE5WREFgLCBgVFNMQWApLgoKLS0tCgojIyDwn4+X77iPIFN5c3RlbSBBcmNoaXRlY3R1cmUKCmBgYG1lcm1haWQKZmxvd2NoYXJ0IExSCiAgICBBWyJSYXcgTmV3cyAmIFByaWNlczxicj4oRk5TUElEIDIzLjIgR0IpIl0gLS0+IEJbIjQ6MDAgUE0gQ3V0b2ZmIEFsaWdubWVudDxicj4oVVMvRWFzdGVybiAmIHBkLm1lcmdlX2Fzb2YpIl0KICAgIEIgLS0+IENbIkZpbkJFUlQgR1BVIEluZmVyZW5jZTxicj4oUHJvc3VzQUkgQmF0Y2ggRXh0cmFjdGlvbikiXQogICAgQiAtLT4gRFsiVGVjaG5pY2FsIEVuZ2luZWVyaW5nPGJyPigxNSBNb21lbnR1bSAmIFZvbGF0aWxpdHkgU2lnbmFscykiXQogICAgQyAmIEQgLS0+IEVbIlVuaWZpZWQgRmVhdHVyZSBGdXNpb248YnI+KG1vZGVsX2RhdGFzZXRfdjEucGFycXVldCkiXQogICAgRSAtLT4gRlsiQ2hyb25vbG9naWNhbCBTcGxpdDxicj4oVHJhaW4gMjAxOC0yMSwgVmFsIDIwMjIsIFRlc3QgMjAyMykiXQogICAgRiAtLT4gR1siT3B0dW5hIEJheWVzaWFuIFR1bmluZzxicj4oNTAtVHJpYWwgVFBFIFJlZ3VsYXJpemF0aW9uKSJdCiAgICBHIC0tPiBIWyJQbGF0dCBDYWxpYnJhdGlvbjxicj4oQ2FsaWJyYXRlZENsYXNzaWZpZXJDVikiXQogICAgSCAtLT4gSVsiVmVjdG9yaXplZCBCYWNrdGVzdGluZzxicj4oMTAgYnBzIE1hcmtldCBGcmljdGlvbikiXQogICAgSCAtLT4gSlsiSW50ZXJhY3RpdmUgRGFzaGJvYXJkPGJyPihTdHJlYW1saXQgRGVtb25zdHJhdGlvbikiXQpgYGAKCjEuICoqUGhhc2UgMeKAkzU6IERhdGEgSW5nZXN0aW9uICYgVGltZS1BbGlnbm1lbnQ6KiogVmVjdG9yaXplZCBjbGVhbmluZyBvZiBPSExDViBwcmljZXMgYW5kIHN0cmVhbWluZyBjaHVua2VkIHByb2Nlc3Npbmcgb2YgMjMuMiBHQiBGTlNQSUQgbmV3cyBoZWFkbGluZXMuIFN0cmljdCB0aW1lc3RhbXAgYWxpZ25tZW50IHNoaWZ0cyBhZnRlci1ob3VycyBuZXdzICgkPiBcdGV4dHs0OjAwIFBNfSQpIHRvIHRoZSBzdWJzZXF1ZW50IHRyYWRlIGRhdGUuCjIuICoqUGhhc2UgNuKAkzE0OiBNdWx0aS1Nb2RhbCBGZWF0dXJlIEZ1c2lvbjoqKiAxNSB0ZWNobmljYWwgaW5kaWNhdG9ycyAoU01BLCBFTUEsIFdpbGRlcidzIFJTSS0xNCwgTUFDRCwgcm9sbGluZyB2b2xhdGlsaXR5KSBmdXNlZCB3aXRoIDggRmluQkVSVCBzZW50aW1lbnQgYWdncmVnYXRlcyAobWVhbiwgc3RkLCBtaW4sIG1heCwgcG9sYXJpdHkgcmF0aW9zKS4KMy4gKipQaGFzZSAxNeKAkzIwOiBDaHJvbm9sb2dpY2FsIFZhbGlkYXRpb246KiogVGVtcG9yYWwgcGFydGl0aW9uaW5nIGludG8gVHJhaW4gKDIwMTjigJMyMDIxLCA4LDI0MSByb3dzKSwgVmFsaWRhdGlvbiAoMjAyMiBiZWFyIG1hcmtldCwgMSw3NTcgcm93cyksIGFuZCBPdXQtb2YtU2FtcGxlIFRlc3QgKDIwMjMgYnVsbCByZWNvdmVyeSwgMSw3MzYgcm93cykuCjQuICoqUGhhc2UgMjHigJMyNDogQmF5ZXNpYW4gVHVuaW5nICYgQ2FsaWJyYXRpb246KiogNTAtdHJpYWwgT3B0dW5hIG9wdGltaXphdGlvbiBvdmVyIGhlYXZpbHkgcmVndWxhcml6ZWQgcGFyYW1ldGVyIHNwYWNlIChgbWF4X2RlcHRoOiAy4oCTNGAsIEwxL0wyIHNocmlua2FnZSksIGZvbGxvd2VkIGJ5IFBsYXR0IFNjYWxpbmcgKFNpZ21vaWQpIG9uIHRoZSAyMDIyIFZhbGlkYXRpb24gc2V0IChgY3Y9J3ByZWZpdCdgKS4KNS4gKipQaGFzZSAyNjogRmluYW5jaWFsIFNpbXVsYXRpb246KiogVmVjdG9yaXplZCBleGVjdXRpb24gZW5naW5lIHdpdGggMTAgYnBzIHNsaXBwYWdlL2NvbW1pc3Npb25zIHBlciB0cmFkZSwgZHluYW1pYyBjb252aWN0aW9uIGh1cmRsZXMsIGFuZCB1bmRlcndhdGVyIGRyYXdkb3duIHRyYWNraW5nLgo2LiAqKlBoYXNlIDI54oCTMzM6IERhc2hib2FyZCAmIFRlc3Rpbmc6KiogRHVhbC10YWIgZGVtb25zdHJhdGlvbiBkYXNoYm9hcmQgYnVpbHQgd2l0aCBTdHJlYW1saXQsIFBsb3RseSwgYW5kIFNIQVAgd2F0ZXJmYWxsIGV4cGxhaW5hYmlsaXR5LCBiYWNrZWQgYnkgYW4gYXV0b21hdGVkIGBweXRlc3RgIHN1aXRlLgoKLS0tCgojIyDwn5SsIEtleSBSZXNlYXJjaCBGaW5kaW5ncwoKfCBNb2RlbCAvIFN0cmF0ZWd5IENvbmZpZ3VyYXRpb24gfCBST0MtQVVDIHwgRjEtU2NvcmUgfCBBbm51YWxpemVkIFJldHVybiB8IFNoYXJwZSBSYXRpbyB8IE1heCBEcmF3ZG93biB8IFdpbiBSYXRlIHwKfCA6LS0tIHwgOi0tLTogfCA6LS0tOiB8IDotLS06IHwgOi0tLTogfCA6LS0tOiB8IDotLS06IHwKfCAqKkJhc2VsaW5lIEE6IE1ham9yaXR5IENsYXNzIENsYXNzaWZpZXIqKiB8IDAuNTAwMCB8IDAuNjgzMCB8IC0gfCAtIHwgLSB8IDUxLjglIHwKfCAqKkJhc2VsaW5lIEI6IFByZXZpb3VzLURheSBNb21lbnR1bSBIZXVyaXN0aWMqKiB8IDAuNTAxMCB8IDAuNTEyMCB8IC0gfCAtIHwgLSB8IDUwLjElIHwKfCAqKkV4cGVyaW1lbnQgQTogUHJpY2UtT25seSBYR0Jvb3N0KiogfCAwLjUxNzkgfCAwLjY5ODAgfCArMzguMiUgfCAxLjYyIHwgLTE1LjQwJSB8IDUzLjYlIHwKfCAqKkV4cGVyaW1lbnQgQjogTmV3cy1Pbmx5IFhHQm9vc3QqKiB8IDAuNTExNSB8IDAuNjgxMCB8ICsyMi40JSB8IDEuMTQgfCAtMTguMjAlIHwgNTEuMiUgfAp8ICoqRXhwZXJpbWVudCBDOiBNdWx0aS1Nb2RhbCBYR0Jvb3N0IChSYXcpKiogfCAwLjU0MzggfCAwLjcxMzAgfCArNDguOSUgfCAyLjEyIHwgLTEyLjMwJSB8IDU2LjQlIHwKfCAqKlR1bmVkICYgQ2FsaWJyYXRlZCBNdWx0aS1Nb2RhbCAoTkVYVVMgQUkpKiogfCAqKjAuNTQ4MioqIHwgKiowLjcxNjAqKiB8ICoqKzU0LjglKiogfCAqKjIuNjgqKiB8ICoqLTkuNzUlKiogfCAqKjU4LjUlKiogfAp8ICpCdXkgJiBIb2xkIEJlbmNobWFyayAoRXF1YWwtV2VpZ2h0KSogfCAqMC41MDAwKiB8ICowLjY4MzAqIHwgKis0OC45JSogfCAqMS44OCogfCAqLTE0LjcwJSogfCAqNTQuMiUqIHwKCiogKipUaGUgTXVsdGktTW9kYWwgQWR2YW50YWdlOioqIEludGVncmF0aW5nIEZpbkJFUlQgZGFpbHkgc2VudGltZW50IHlpZWxkcyBhbiBhYnNvbHV0ZSBST0MtQVVDIGltcHJvdmVtZW50IG9mICoqKzAuMDI1OSAoKzUuMjYlIHJlbGF0aXZlIGdhaW4pKiogb3ZlciB0ZWNobmljYWwgcHJpY2UgaW5kaWNhdG9ycyBhbG9uZS4gU3RhbmRhbG9uZSBzZW50aW1lbnQgZXhoaWJpdHMgbG93IGlzb2xhdGVkIHByZWRpY3RhYmlsaXR5LCBidXQgZnVuY3Rpb25zIGFzIGEgcG90ZW50ICoqY29udGV4dHVhbCBmaWx0ZXIqKiB3aGVuIGNvdXBsZWQgd2l0aCBtb21lbnR1bS4KKiAqKkNhcGl0YWwgUHJlc2VydmF0aW9uICYgRHJhd2Rvd24gUmVkdWN0aW9uOioqIFRoZSBEeW5hbWljIE1lZGlhbiBzdHJhdGVneSBhY2hpZXZlZCBhICoqNTQuOCUgQ0FHUioqIHdpdGggYSAqKjU4LjUlIHdpbiByYXRlKiosIHJlZHVjaW5nIG1heGltdW0gZHJhd2Rvd24gZnJvbSAqKi0xNC43MCUgKEJlbmNobWFyaykgdG8gLTkuNzUlIChORVhVUyBBSSkqKiBhZnRlciBkZWR1Y3RpbmcgMTAgYnBzIHRyYW5zYWN0aW9uIGZyaWN0aW9uIG9uIGFsbCBwb3NpdGlvbiBjaGFuZ2VzLgoKLS0tCgojIyDwn5qAIFF1aWNrc3RhcnQgJiBTZXR1cAoKIyMjIDEuIENsb25lICYgU2V0IFVwIEVudmlyb25tZW50CgpgYGBiYXNoCmdpdCBjbG9uZSBodHRwczovL2dpdGh1Yi5jb20veW91ci11c2VybmFtZS9ORVhVU19TdG9ja19BSS5naXQKY2QgTkVYVVNfU3RvY2tfQUkKCiMgQ3JlYXRlIGFuZCBhY3RpdmF0ZSB2aXJ0dWFsIGVudmlyb25tZW50CnB5dGhvbjMgLW0gdmVudiAudmVudgpzb3VyY2UgLnZlbnYvYmluL2FjdGl2YXRlCgojIEluc3RhbGwgZGVwZW5kZW5jaWVzCnBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy50eHQKYGBgCgojIyMgMi4gUnVuIEF1dG9tYXRlZCBWZXJpZmljYXRpb24gVGVzdHMKCmBgYGJhc2gKcHl0ZXN0IHRlc3RzLyAtdgpgYGAKClRoZSB0ZXN0IHN1aXRlIHZhbGlkYXRlczoKLSAqKk5vIERhdGEgTGVha2FnZToqKiBUYXJnZXQgYW5kIGZ1dHVyZSByZXR1cm4gY29sdW1ucyBhcmUgc3RyaWN0bHkgZXhjbHVkZWQgZnJvbSBgZmVhdHVyZV9zY2hlbWEuanNvbmAuCi0gKipUZW1wb3JhbCBEaXNqb2ludG5lc3M6KiogVHJhaW5pbmcsIHZhbGlkYXRpb24sIGFuZCB0ZXN0IHdpbmRvd3MgaGF2ZSB6ZXJvIGRhdGUgb3ZlcmxhcC4KLSAqKkRhdGEgSW50ZWdyaXR5OioqIE5vIGR1cGxpY2F0ZSBgKHN0b2NrX3N5bWJvbCwgZGF0ZSlgIHBhaXJzIGV4aXN0IGluIGNsZWFuZWQgcHJpY2UgZGF0YS4KLSAqKkluZmVyZW5jZSBFbmdpbmU6KiogVmFsaWRhdGVzIHN0YW5kYWxvbmUgZXhlY3V0aW9uIGFuZCBTSEFQIGZhY3RvciBhdHRyaWJ1dGlvbi4KCiMjIyAzLiBMYXVuY2ggSW50ZXJhY3RpdmUgU3RyZWFtbGl0IERhc2hib2FyZAoKYGBgYmFzaApzdHJlYW1saXQgcnVuIHN0cmVhbWxpdF9hcHAvYXBwLnB5CmBgYAoKT3BlbiBgaHR0cDovL2xvY2FsaG9zdDo4NTAxYCB0byBleHBsb3JlOgotICoqVGFiIDE6KiogTGl2ZSBkaXJlY3Rpb25hbCBwcmVkaWN0aW9ucywgcHJvYmFiaWxpdHkgbWV0ZXJzLCBTSEFQIHByZWRpY3RpdmUgZHJpdmVycywgUGxvdGx5IGludGVyYWN0aXZlIGNhbmRsZXN0aWNrIGNoYXJ0cywgYW5kIHNlc3Npb24gaGVhZGxpbmUgc2VudGltZW50IGZlZWRzLgotICoqVGFiIDI6KiogTW9kZWwgbGVhZGVyYm9hcmQsIGFibGF0aW9uIHN0dWRpZXMsIGFuZCBkaWFnbm9zdGljIFJPQyAvIFNIQVAgc3VtbWFyeSBwbG90cy4KCiMjIyA0LiBTdGFuZGFsb25lIFB5dGhvbiBJbmZlcmVuY2UKCmBgYHB5dGhvbgpmcm9tIHNyYy5pbmZlcmVuY2UucHJlZGljdCBpbXBvcnQgTmV4dXNJbmZlcmVuY2VFbmdpbmUKaW1wb3J0IHBhbmRhcyBhcyBwZAoKIyBMb2FkcyBjYWxpYnJhdGVkIG1vZGVsIGFuZCBzY2hlbWEgYXV0b21hdGljYWxseQplbmdpbmUgPSBOZXh1c0luZmVyZW5jZUVuZ2luZSgpCgojIFBhc3MgYSBEYXRhRnJhbWUgY29udGFpbmluZyB0aGUgMjMgcmVxdWlyZWQgdGVjaG5pY2FsICYgc2VudGltZW50IGZlYXR1cmVzCnJlc3VsdHMgPSBlbmdpbmUucHJlZGljdChzYW1wbGVfZmVhdHVyZXNfZGYsIHRvcF9rX2ZhY3RvcnM9MykKcHJpbnQocmVzdWx0cykKIyBPdXRwdXQ6IFt7J3RpY2tlcic6ICdOVkRBJywgJ3ByZWRpY3Rpb24nOiAnVVAnLCAndXBfcHJvYmFiaWxpdHknOiAwLjYyMTQsICd0b3BfZmFjdG9ycyc6IFsuLi5dfV0KYGBgCgotLS0KCiMjIPCfk4IgUmVwb3NpdG9yeSBTdHJ1Y3R1cmUKCmBgYHRleHQK4pSc4pSA4pSAIGRhdGEvICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBQYXJxdWV0IGRhdGFzZXRzICYgc2NoZW1hcwrilIIgICDilJzilIDilIAgbW9kZWxfZGF0YXNldF92MS5wYXJxdWV0ICAgICAgICAgICMgMjMtZmVhdHVyZSB1bmlmaWVkIE1MIGRhdGFzZXQK4pSCICAg4pSU4pSA4pSAIHNlbnRpbWVudF9uZXdzLnBhcnF1ZXQgICAgICAgICAgICAjIEZpbkJFUlQgY2xhc3NpZmllZCBuZXdzIGhlYWRsaW5lcwrilJzilIDilIAgbW9kZWxzLyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIFNlcmlhbGl6ZWQgd2VpZ2h0cyAmIG1ldGFkYXRhCuKUgiAgIOKUnOKUgOKUgCB4Z2JfZGlyZWN0aW9uX2NhbGlicmF0ZWQuam9ibGliICAgIyBDaGFtcGlvbiBQbGF0dC1jYWxpYnJhdGVkIHBpcGVsaW5lCuKUgiAgIOKUnOKUgOKUgCB4Z2JfZGlyZWN0aW9uLmpzb24gICAgICAgICAgICAgICAgIyBOYXRpdmUgWEdCb29zdCBib29zdGVyIHJlZmVyZW5jZQrilIIgICDilJzilIDilIAgZmVhdHVyZV9zY2hlbWEuanNvbiAgICAgICAgICAgICAgICMgMjMtZmVhdHVyZSBzY2hlbWEgZGVmaW5pdGlvbgrilIIgICDilJzilIDilIAgbW9kZWxfbWV0YWRhdGEuanNvbiAgICAgICAgICAgICAgICMgRnVsbCBoeXBlcnBhcmFtZXRlcnMgJiB0ZXN0IG1ldHJpY3MK4pSCICAg4pSc4pSA4pSAIGJhY2t0ZXN0X3Jlc3VsdHMuanNvbiAgICAgICAgICAgICAjIFBoYXNlIDI2IGJhY2t0ZXN0IHN1bW1hcnkK4pSCICAg4pSc4pSA4pSAIHNoYXBfc3VtbWFyeV9wbG90LnBuZyAgICAgICAgICAgICAjIEdsb2JhbCBTSEFQIGZlYXR1cmUgaW1wb3J0YW5jZQrilIIgICDilJTilIDilIAgYmFja3Rlc3RfZXF1aXR5X2N1cnZlLnBuZyAgICAgICAgICMgMi1wYW5lbCBiYWNrdGVzdCBlcXVpdHkgJiBkcmF3ZG93biBjaGFydArilJzilIDilIAgc3JjLyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIE1vZHVsYXIgcHJvZHVjdGlvbiBzb3VyY2UgY29kZQrilIIgICDilJzilIDilIAgaW5mZXJlbmNlLwrilIIgICDilIIgICDilJzilIDilIAgX19pbml0X18ucHkK4pSCICAg4pSCICAg4pSU4pSA4pSAIHByZWRpY3QucHkgICAgICAgICAgICAgICAgICAgICMgU3RhbmRhbG9uZSBOZXh1c0luZmVyZW5jZUVuZ2luZQrilIIgICDilJTilIDilIAgYmFja3Rlc3QvCuKUgiAgICAgICDilJzilIDilIAgX19pbml0X18ucHkK4pSCICAgICAgIOKUlOKUgOKUgCBlbmdpbmUucHkgICAgICAgICAgICAgICAgICAgICAjIFZlY3Rvcml6ZWRCYWNrdGVzdGVyCuKUnOKUgOKUgCBzdHJlYW1saXRfYXBwLwrilIIgICDilJTilIDilIAgYXBwLnB5ICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgRGVtb25zdHJhdGlvbiBkYXNoYm9hcmQgVUkK4pSc4pSA4pSAIHRlc3RzLwrilIIgICDilJzilIDilIAgX19pbml0X18ucHkK4pSCICAg4pSU4pSA4pSAIHRlc3RfcGlwZWxpbmUucHkgICAgICAgICAgICAgICAgICAjIFB5dGVzdCBsZWFrYWdlICYgaW50ZWdyaXR5IHN1aXRlCuKUnOKUgOKUgCBsYXVuY2hfc3RyZWFtbGl0LmlweW5iICAgICAgICAgICAgICAgICMgU2VsZi1jb250YWluZWQgQ29sYWIgZGFlbW9uIGxhdW5jaGVyCuKUnOKUgOKUgCBwaGFzZTFfZm5zcGlkX2luZ2VzdGlvbi5pcHluYiAgICAgICAgICMgUGhhc2UgMTogMjMgR0Igc3RyZWFtaW5nIGluZ2VzdGlvbgrilJzilIDilIAgcGhhc2UyX3RvXzRfY2xlYW5fYWxpZ24uaXB5bmIgICAgICAgICAjIFBoYXNlcyAyLTQ6IDQ6MDAgUE0gY3V0b2ZmIGFsaWdubWVudArilJzilIDilIAgcGhhc2U3X3RvXzlfZmluYmVydF9zZW50aW1lbnQuaXB5bmIgICAjIFBoYXNlcyA3LTk6IEJhdGNoIEdQVSBzZW50aW1lbnQgc2NvcmluZwrilJzilIDilIAgcGhhc2U2X3RvXzE0X2RhdGFzZXRfZW5naW5lZXJpbmcuaXB5bmIjIFBoYXNlcyA2LCAxMC0xNDogTXVsdGktbW9kYWwgZnVzaW9uCuKUnOKUgOKUgCBwaGFzZTE1X3RvXzIwX21vZGVsX2V4cGVyaW1lbnRzLmlweW5iICMgUGhhc2VzIDE1LTIwOiBYR0Jvb3N0IGJlbmNobWFya3MK4pSc4pSA4pSAIHBoYXNlMjFfdG9fMjhfZXhwbGFpbl9hbmRfaW5mZXJlbmNlLmlweW5iICMgUGhhc2VzIDIxLTI4OiBTSEFQICYgU2VyaWFsaXphdGlvbgrilJzilIDilIAgcGhhc2UyM18yNF90dW5pbmdfYW5kX2NhbGlicmF0aW9uLmlweW5iICMgUGhhc2VzIDIzLTI0OiBPcHR1bmEgJiBQbGF0dCBTY2FsaW5nCuKUnOKUgOKUgCBwaGFzZTI2X2JhY2t0ZXN0aW5nX3NpbXVsYXRpb24uaXB5bmIgICMgUGhhc2UgMjY6IFZlY3Rvcml6ZWQgYmFja3Rlc3RpbmcK4pSc4pSA4pSAIHBoYXNlMzFfdG9fMzNfZmluYWxfcGFja2FnaW5nLmlweW5iICAgIyBQaGFzZXMgMzEtMzM6IFRlc3RpbmcgJiBkb2N1bWVudGF0aW9uCuKUnOKUgOKUgCByZXF1aXJlbWVudHMudHh0ICAgICAgICAgICAgICAgICAgICAgICMgUHJvamVjdCBkZXBlbmRlbmN5IGxvY2tmaWxlCuKUlOKUgOKUgCBSRUFETUUubWQgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgQ29tcHJlaGVuc2l2ZSBwcm9qZWN0IGRvY3VtZW50YXRpb24KYGBgCgotLS0KCiMjIOKalu+4jyBMaWNlbnNlICYgRGlzY2xhaW1lcgoKRGlzdHJpYnV0ZWQgdW5kZXIgdGhlIE1JVCBMaWNlbnNlLiBUaGlzIHNvZnR3YXJlIGlzIGJ1aWx0IGZvciBlZHVjYXRpb25hbCBhbmQgcmVzZWFyY2ggcHVycG9zZXMuIEl0IGRvZXMgbm90IGNvbnN0aXR1dGUgZmluYW5jaWFsLCBpbnZlc3RtZW50LCBvciB0cmFkaW5nIGFkdmljZS4gUGFzdCBtb2RlbCBwZXJmb3JtYW5jZSBpcyBub3QgaW5kaWNhdGl2ZSBvZiBmdXR1cmUgbWFya2V0IHJldHVybnMuCg=="
with open("README.md", "wb") as f:
    f.write(base64.b64decode(readme_b64))
print(f"✓ Created root README.md ({os.path.getsize('README.md')} bytes)")

if os.path.exists(GDRIVE_BASE):
    shutil.copy2("README.md", os.path.join(GDRIVE_BASE, "README.md"))
    print("✓ Synced README.md to Google Drive.")


PHASE 33: GENERATING PRODUCTION README.md
✓ Created root README.md (9439 bytes)
✓ Synced README.md to Google Drive.


### 📦 Step 5: Final Deployment Packaging (`nexus_data_backup.zip`)
Packages all production modules, models, tests, requirements, and documentation into the definitive archive.

In [5]:
print("=" * 80)
print("FINAL DEPLOYMENT PACKAGING")
print("=" * 80)

target_zip = "/content/nexus_data_backup.zip" if os.path.exists("/content") else "./nexus_data_backup.zip"
print(f"Building definitive project archive: {target_zip}...")

with zipfile.ZipFile(target_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Package folders
    for folder in ["./streamlit_app", "./src", "./models", "./data", "./tests"]:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    if not file.endswith(".pyc") and "__pycache__" not in root:
                        fpath = os.path.join(root, file)
                        arcname = os.path.relpath(fpath, ".")
                        zipf.write(fpath, arcname)
    
    # Package standalone files
    for standalone_file in ["README.md", "requirements.txt"]:
        if os.path.exists(standalone_file):
            zipf.write(standalone_file, arcname=standalone_file)

zip_size = os.path.getsize(target_zip) / (1024 * 1024)
print(f"✓ Final deployment archive created successfully ({zip_size:.2f} MB).")

# Sync to Drive
if os.path.exists(GDRIVE_BASE):
    gdrive_zip = os.path.join(GDRIVE_BASE, "nexus_data_backup.zip")
    shutil.copy2(target_zip, gdrive_zip)
    print(f"✓ Synced definitive archive to Google Drive: {gdrive_zip}")


FINAL DEPLOYMENT PACKAGING
Building definitive project archive: /content/nexus_data_backup.zip...
✓ Final deployment archive created successfully (277.15 MB).
✓ Synced definitive archive to Google Drive: /content/drive/MyDrive/NEXUS_Stock_AI/nexus_data_backup.zip
